In [11]:
# Verificação do schema da tabela bronze

df = spark.sql("SELECT * FROM emendas_parlamentares_bronze")
df.printSchema()

root
 |-- Código_da_Emenda: string (nullable = true)
 |-- Ano_da_Emenda: integer (nullable = true)
 |-- Tipo_de_Emenda: string (nullable = true)
 |-- Código_do_Autor_da_Emenda: string (nullable = true)
 |-- Nome_do_Autor_da_Emenda: string (nullable = true)
 |-- Número_da_emenda: string (nullable = true)
 |-- Localidade_de_aplicação_do_recurso: string (nullable = true)
 |-- Código_Município_IBGE: string (nullable = true)
 |-- Município: string (nullable = true)
 |-- Código_UF_IBGE: integer (nullable = true)
 |-- UF: string (nullable = true)
 |-- Região: string (nullable = true)
 |-- Código_Função: string (nullable = true)
 |-- Nome_Função: string (nullable = true)
 |-- Código_Subfunção: string (nullable = true)
 |-- Nome_Subfunção: string (nullable = true)
 |-- Código_Programa: string (nullable = true)
 |-- Nome_Programa: string (nullable = true)
 |-- Código_Ação: string (nullable = true)
 |-- Nome_Ação: string (nullable = true)
 |-- Código_Plano_Orçamentário: string (nullable = true)
 

In [ ]:
# Ajuste de tipos e valores inválidos

from pyspark.sql import functions as F

placeholders = {
    "Código_da_Emenda": ("Sem informação",),
    "Código_do_Autor_da_Emenda": ("S/I",),
    "Número_da_emenda": ("S/I",),
    "Código_Município_IBGE": ("Sem informação", "S/I"),
    "Nome_do_Autor_da_Emenda": ("Sem informação",),
    "Localidade_de_aplicação_do_recurso": ("Sem informação",),
    "Município": ("Sem informação",),
    "UF": ("Sem informação",),
    "Código_Função": ("MU",),
    "Código_Subfunção": ("MU",),
    "Código_Programa": ("MU",),
}

for nome, valores_invalidos in placeholders.items():
    valor = F.trim(F.col(nome))
    df = df.withColumn(
        nome,
        F.when(valor.isin(*valores_invalidos), F.lit(None).cast("string"))
        .otherwise(valor.cast("string"))
    )

df = (
    df
    .withColumn("Ano_da_Emenda", F.col("Ano_da_Emenda").cast("int"))
    .withColumn(
        "Código_Município_IBGE",
        F.when(
            F.trim(F.col("Código_Município_IBGE")).rlike(r"^-\d+$"),
            F.lit(None).cast("string")
        ).otherwise(F.trim(F.col("Código_Município_IBGE")).cast("string"))
    )
    .withColumn(
        "Código_UF_IBGE",
        F.when(F.col("Código_UF_IBGE") < 0, None)
        .otherwise(F.col("Código_UF_IBGE").cast("int"))
    )
)

def valor_brl(nome):
    valor = F.trim(F.col(nome))
    valor = F.regexp_replace(valor, r"\.", "")
    valor = F.regexp_replace(valor, ",", ".")

    return F.when(
        valor.rlike(r"^-?\d+(\.\d{1,2})?$"),
        valor.cast("decimal(18,2)")
    ).otherwise(F.lit(None).cast("decimal(18,2)"))

colunas_valor = (
    "Valor_Empenhado",
    "Valor_Liquidado",
    "Valor_Pago",
    "Valor_Restos_A_Pagar_Inscritos",
    "Valor_Restos_A_Pagar_Cancelados",
    "Valor_Restos_A_Pagar_Pagos",
)

for nome in colunas_valor:
    df = df.withColumn(nome, valor_brl(nome))

df.printSchema()

In [ ]:

tabela = "emendas_parlamentares_prata"

df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(tabela)